# Os quatro pilares da OO

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Proteger o estado de um objeto com campos private e métodos que conferem cada mudança (encapsulamento).
- Construir uma classe em cima de outra com extends e super, e sobrescrever os métodos dela (herança).
- Tratar objetos de classes diferentes pelo tipo do pai, cada um respondendo do seu jeito (polimorfismo).

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Os quatro pilares
2. Encapsulamento: campos private
3. Herança: extends e super
4. Sobrescrita e polimorfismo
5. Abstração
6. Erros comuns

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## 1. Os quatro pilares

A programação orientada a objetos se apoia em quatro ideias. Você já usou as duas primeiras um pouco; esta lição passa por cada uma.

| Pilar | Significa | Em Java |
|---|---|---|
| **Encapsulamento** | um objeto protege o próprio estado | campos `private`, mudados só por métodos |
| **Herança** | uma classe é construída sobre outra | `class Dog extends Animal` |
| **Polimorfismo** | uma chamada, uma resposta diferente para cada tipo de objeto | sobrescrever métodos, uma variável do tipo do pai |
| **Abstração** | o código trabalha com a ideia geral, não com os detalhes de cada tipo | um método ou tipo que esconde como as coisas são feitas |

## 2. Encapsulamento: campos private

Se qualquer código pode mudar um campo, nada impede um valor inválido, como um termostato a 500 graus. Marque os campos como `private`: só os métodos da própria classe podem mexer neles. O resto do código passa por métodos que conferem cada mudança. Um método que só lê um campo costuma se chamar **getter**.

In [ ]:
class Thermostat {
    private int target = 20;

    void setTarget(int degrees) {
        if (degrees < 10 || degrees > 30) {
            IO.println("Ignorado: " + degrees + " está fora da faixa");
            return;
        }
        target = degrees;
    }

    int getTarget() {
        return target;
    }
}

void main() {
    Thermostat t = new Thermostat();
    t.setTarget(23);
    t.setTarget(500);
    IO.println("Alvo: " + t.getTarget());
}

🧪 **Experimente:** no `main`, escreva `t.target = 500;` e leia o erro: o `private` impede. Depois acrescente `void warmer()`, que sobe o alvo em 1, mas nunca acima de 30.

## 3. Herança: extends e super

`class Dog extends Animal` faz Dog ser uma **subclasse** de Animal: ela herda os campos e métodos de Animal e pode acrescentar os seus. O construtor dela precisa montar primeiro a parte de Animal, com `super(...)` na primeira linha. Os campos `private` do pai continuam privados: a subclasse os alcança pelos métodos do pai.

In [ ]:
class Animal {
    private final String name;

    Animal(String name) {
        this.name = name;
    }

    String name() {
        return name;
    }
}

class Dog extends Animal {
    private final String breed;

    Dog(String name, String breed) {
        super(name);              // monta primeiro a parte de Animal
        this.breed = breed;
    }

    String card() {
        return name() + ", um " + breed;   // name() é herdado
    }
}

void main() {
    Dog rex = new Dog("Rex", "beagle");
    IO.println(rex.card());
    IO.println(rex.name());
}

🧪 **Experimente:** tire a linha `super(name);` e leia o erro. Depois, em `card()`, use o campo `name` direto em vez de `name()`: ele é privado de Animal.

### ✏️ Exercício: Um contador que conta em passos

O Counter soma 1. Faça o StepCounter sobrescrever `add()` para somar `step`, chamando o `add()` do pai com `super.add()` esse número de vezes.

In [ ]:
class Counter {
    private int value;

    void add() {
        value++;
    }

    int value() {
        return value;
    }
}

class StepCounter extends Counter {
    private final int step;

    StepCounter(int step) {
        this.step = step;
    }

    @Override
    void add() {
        // TODO: chame o add() do pai `step` vezes, com super.add()
    }
}

void main() {
    StepCounter three = new StepCounter(3);
    three.add();
    three.add();
    check("dois adds de 3", three.value(), 6);
    StepCounter one = new StepCounter(1);
    one.add();
    check("um passo de 1", one.value(), 1);
}

<details>
<summary>Ver uma resposta</summary>

```java
class Counter {
    private int value;

    void add() {
        value++;
    }

    int value() {
        return value;
    }
}

class StepCounter extends Counter {
    private final int step;

    StepCounter(int step) {
        this.step = step;
    }

    @Override
    void add() {
        for (int i = 0; i < step; i++) {
            super.add();
        }
    }
}

void main() {
    StepCounter three = new StepCounter(3);
    three.add();
    three.add();
    check("dois adds de 3", three.value(), 6);
    StepCounter one = new StepCounter(1);
    one.add();
    check("um passo de 1", one.value(), 1);
}
```

</details>

## 4. Sobrescrita e polimorfismo

Uma subclasse pode **sobrescrever** um método: escrever um método com o mesmo nome e parâmetros, marcado com `@Override`. Aí uma variável do tipo do pai pode guardar qualquer subclasse, e chamar o método roda a versão da classe do **objeto**, escolhida enquanto o programa roda. Isso é **polimorfismo**: uma linha de código, uma resposta diferente para cada tipo de objeto. Os slides acompanham isso.

#### 🎞️ Qual sound() roda?

![Uma List<Animal> guarda um Dog e um Animal; describe() é herdado de Animal e chama sound(), e o Java roda a versão da classe de cada objeto: Au para o Dog, ... para o Animal](dispatch-steps.pt-br.svg)

In [ ]:
class Animal {
    private final String name;

    Animal(String name) {
        this.name = name;
    }

    String describe() {
        return name + ": " + sound();
    }

    String sound() {
        return "...";
    }
}

class Dog extends Animal {
    Dog(String name) {
        super(name);
    }

    @Override
    String sound() {
        return "Au";
    }
}

class Cat extends Animal {
    Cat(String name) {
        super(name);
    }

    @Override
    String sound() {
        return "Miau";
    }
}

void main() {
    List<Animal> zoo = List.of(new Dog("Rex"), new Cat("Mia"), new Animal("Ann"));
    for (Animal a : zoo) {
        IO.println(a.describe());
    }
}

🧪 **Experimente:** acrescente uma `Cow` e ponha na lista: o laço não precisa mudar. Depois escreva `sound` errado em Cat, como `Sound`, mantendo o `@Override`, e leia o erro.

### ✏️ Exercício: Rodas

Faça a Bike sobrescrever `wheels()` para devolver 2, e escreva `totalWheels`, que soma as rodas de todos os veículos da lista. O laço chama `wheels()` sem saber de que classe é cada veículo.

In [ ]:
class Vehicle {
    int wheels() {
        return 4;
    }
}

class Bike extends Vehicle {
    // TODO: sobrescreva wheels() para devolver 2
}

int totalWheels(List<Vehicle> vehicles) {
    // TODO: some v.wheels() de todos os veículos
    return 0;
}

void main() {
    check("uma bike", new Bike().wheels(), 2);
    check("um carro e duas bikes", totalWheels(List.of(new Vehicle(), new Bike(), new Bike())), 8);
    check("nenhum veículo", totalWheels(List.of()), 0);
}

<details>
<summary>Ver uma resposta</summary>

```java
class Vehicle {
    int wheels() {
        return 4;
    }
}

class Bike extends Vehicle {
    @Override
    int wheels() {
        return 2;
    }
}

int totalWheels(List<Vehicle> vehicles) {
    int total = 0;
    for (Vehicle v : vehicles) {
        total += v.wheels();
    }
    return total;
}

void main() {
    check("uma bike", new Bike().wheels(), 2);
    check("um carro e duas bikes", totalWheels(List.of(new Vehicle(), new Bike(), new Bike())), 8);
    check("nenhum veículo", totalWheels(List.of()), 0);
}
```

</details>

## 5. Abstração

**Abstração** quer dizer que o código trabalha com a ideia geral e deixa os detalhes para cada tipo. O `describe()` acima nunca pergunta "é um cachorro?": ele só chama `sound()`. Um método que recebe um `Animal` funciona para todos os animais, até para tipos escritos depois. A próxima lição leva isso adiante: uma **classe abstrata** ou uma **interface** descreve a ideia sem nenhuma versão padrão.

In [ ]:
class Animal {
    String sound() {
        return "...";
    }
}

class Dog extends Animal {
    @Override
    String sound() {
        return "Au";
    }
}

String chorus(List<Animal> animals) {       // só conhece a ideia geral
    StringBuilder sb = new StringBuilder();
    for (Animal a : animals) {
        sb.append(a.sound()).append(' ');
    }
    return sb.toString().trim();
}

void main() {
    IO.println(chorus(List.of(new Dog(), new Dog(), new Animal())));
}

🧪 **Experimente:** escreva uma subclasse `Duck` e use-a no `chorus` sem mexer no `chorus`.

## 6. Erros comuns

- **Um erro de digitação em vez de sobrescrita:** sem o `@Override`, `String Sound()` é um método novo, e o `sound()` do pai continua rodando.
- **`super(...)` fora da primeira linha:** a parte do pai precisa ser montada antes de tudo no construtor.
- **Alcançar um campo `private` do pai:** use os métodos do pai.
- **Chamar um método que só a subclasse tem pelo tipo do pai:** uma variável `Animal` só oferece os métodos de Animal, mesmo guardando um Dog.

In [ ]:
class Animal {
    String sound() {
        return "...";
    }
}

class Dog extends Animal {
    String Sound() {              // erro de digitação: isto não substitui sound()
        return "Au";
    }

    void fetch() {
        IO.println("buscando!");
    }
}

void main() {
    Animal a = new Dog();
    IO.println(a.sound());        // ainda "...": nada sobrescreveu
    Dog d = new Dog();
    d.fetch();
}

🧪 **Experimente:** ponha `@Override` acima de `String Sound()` e leia o erro. Conserte o nome. Depois tente `a.fetch();`: uma variável `Animal` não oferece esse método.

## Em resumo

A orientação a objetos tem quatro pilares:

| Pilar | Significa |
|---|---|
| **Encapsulamento** | os campos são `private`; os métodos cuidam de cada mudança |
| **Herança** | `class Dog extends Animal` reaproveita e estende uma classe pai |
| **Polimorfismo** | uma variável `Animal` pode guardar qualquer subclasse, e cada uma responde do seu jeito |
| **Abstração** | o código trabalha com a ideia geral (`Animal`), não com os detalhes de cada tipo |

```java
class Animal {
    private final String name;

Animal(String name) {
        this.name = name;
    }

String describe() {
        return name + ": " + sound();
    }

String sound() {
        return "...";
    }
}

class Dog extends Animal {
    Dog(String name) {
        super(name);
    }

@Override
    String sound() {
        return "Au";
    }
}

void main() {
    List<Animal> zoo = List.of(new Dog("Rex"), new Animal("Genérico"));
    for (Animal a : zoo) {
        IO.println(a.describe());
    }
}
```

O `describe()` chama `sound()`, e o Java escolhe a versão da classe do **objeto** enquanto o programa roda.

**Cuidado:** o `@Override` pede ao compilador para conferir que você realmente substitui um método do pai. Sem ele, um erro de digitação como `Sound()` cria um método novo sem avisar.

## Leituras

- [Polymorphism](https://dev.java/learn/language/oop/inheritance/polymorphism/) (dev.java (Oracle), 7 min): O exemplo da Bicycle: cada subclasse sobrescreve printDescription, e a mesma chamada dá uma resposta diferente. Depois o super para chamar a versão do pai, e como os construtores das subclasses chamam o do pai.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Conta bancária (encapsulamento)** · Médio
- **Formas (herança)** · Médio
- **Folha de pagamento (polimorfismo)** · Difícil